# 第 12 章 消費と投資の理論: 数字を変えて確かめる

**使い方**: 「Parameters」のセルの数字を書き換えて、上から順に実行します（メニューの Run → Run All Cells）。数字を変えるたびに、結果と図が変わります。

起動: アプリのディレクトリで `uv run --with jupyterlab jupyter lab notebooks/` を実行します。

記事: `docs/article/macro-basics-programming/12-consumption-and-investment.md`

In [ ]:
# --- Parameters: ここの数字を書き換えます ---
incomes = [100, 100, 100, 130, 100, 100]  # 所得の系列（4 期目だけ一時的に増える）
mpc = 0.8                  # 限界消費性向 c
adjust = 0.2               # 恒常所得の調整係数 λ
annual_income = 500        # 働いている間の年収
working_years = 40         # 働く年数
lifetime_years = 60        # 生涯の年数
capital_coefficient = 2    # 資本係数 V
accelerator_incomes = [100, 120, 140, 150, 155, 155]  # 加速度原理の所得の系列
market_value = 120         # 企業の市場価値
replacement_cost = 100     # 資本の再取得価格

In [ ]:
# --- 準備（変更しません）: lib を読み込めるようにします ---
import sys
from pathlib import Path

_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lib").is_dir())
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

## 消費の理論

In [ ]:
from lib.chapter12.consumption import (
    consumption_from_absolute_income,
    lifecycle_consumption,
    lifecycle_saving,
    permanent_consumption,
    ratchet_consumption,
)

print("絶対所得仮説:", [consumption_from_absolute_income(y, mpc) for y in incomes])
print("恒常所得仮説:",
      [round(c, 1) for c in permanent_consumption(incomes, mpc, adjust)])
print("相対所得仮説（ラチェット）:", ratchet_consumption(incomes, mpc))
print(f"ライフサイクル仮説: 毎年の消費 "
      f"{lifecycle_consumption(annual_income, working_years, lifetime_years):.1f}、"
      f"貯蓄 {lifecycle_saving(annual_income, working_years, lifetime_years):.1f}")

## 図: 消費の理論

In [ ]:
from lib.chapter12.plot import plot_consumption_theories

plot_consumption_theories(incomes, mpc, adjust)

## 投資の理論

In [ ]:
from lib.chapter12.investment import accelerator_path, should_invest, tobins_q

print("加速度原理の投資:", accelerator_path(capital_coefficient, accelerator_incomes))
q = tobins_q(market_value, replacement_cost)
print(f"トービンの q: {q:.2f}、投資するか: {should_invest(q)}")

## 図: 加速度原理

In [ ]:
from lib.chapter12.plot import plot_accelerator

plot_accelerator(capital_coefficient, accelerator_incomes)

## 試してみよう

- `adjust = 0.5` にすると、恒常所得仮説の消費は [80, 80, 80, 92, 86, 83] になります
  （0.2 のときは [80, 80, 80, 84.8, 83.8, 83.1]）。`adjust = 1.0` にすると、
  絶対所得仮説と
  同じ [80, 80, 80, 104, 80, 80] です。**一時的な所得の増加に、消費は敏感に動きません**
- ラチェット（相対所得仮説）の消費は、所得が戻っても、[80, 80, 80, 104, 104, 104] のまま
  下がりません。過去の最高所得が、消費を決めるからです
- `working_years = 45` にすると、毎年の消費は 375.0 になります（40 年では 333.3）。
  働く期間が長いほど、生涯所得が増えます
- `capital_coefficient = 3` にすると、投資は [60, 60, 30, 15, 0] になります
  （2 のときは [40, 40, 20, 10, 0]）。所得の増加が止まる（`accelerator_incomes` の最後を
  150 に揃える）と、投資は 0 になります
- `market_value = 80` にすると、q は 0.80 で、投資しません（q > 1 のときだけ投資します）